In [ ]:
# [env] load .env (KEY=VALUE lines) into os.environ -- see .env.sample
# real env vars / Colab / Kaggle secrets take precedence; empty values are ignored
import os, pathlib
for _p in [pathlib.Path(".env"), pathlib.Path("../.env"),
           pathlib.Path("/content/drive/MyDrive/.env"),
           *sorted(pathlib.Path("/kaggle/input").glob("*/.env"))]:
    if _p.is_file():
        for _line in _p.read_text().splitlines():
            _k, _sep, _v = _line.partition("=")
            _k, _v = _k.strip(), _v.strip().strip("\"'")
            if _sep and _k and _v and not _k.startswith("#"):
                os.environ.setdefault(_k, _v)
        print("env loaded from", _p)
        break
else:
    print("no .env found -- copy .env.sample to .env and fill it in")

# Build the offline asset datasets (run on Kaggle with **Internet ON**, CPU is enough)

The `*_kaggle.ipynb` notebooks run with Internet OFF. Everything they would download is packed here into
Kaggle Datasets: model checkpoints (plain files under `repos/<owner>__<name>/`), pip wheels and torch-hub weights.

A notebook's saved output is capped at about 20 GB, so the assets are split into **bundles**. For each bundle:

1. Set `BUNDLE` below, then **Save Version → Save & Run All (Commit)**.
2. When the run finishes, open the version's **Output** tab → **New Dataset**, and name it `sar-assets-<bundle>` (e.g. `sar-assets-base`).
3. Attach the datasets to the Kaggle notebooks as inputs. The setup cell finds every `repos/`, `wheels/` and `torch/` folder under `/kaggle/input` automatically.

| Bundle | Contents | Needed by |
|---|---|---|
| `base` (~12 GB) | SD-1.5 (vae, unet, text encoder, tokenizer), `adapted16_final`, `bridge_final`, `regressor_v3_final` (+ tuned decoder), `data_meta`, wheels, LPIPS/FID weights | all three notebooks |
| `regressors` (~10 GB) | `regressor_final`, `regressor_v2_final`, `afcrps_final` | SAR-to-Optic (v1/v2, calibration) |
| `flux` (~4 GB) | `regressor_modernvae_final` + FLUX.1-schnell VAE (accept its licence on huggingface.co first) | SAR-to-Optic cell 74 |
| `fyp_runs` (~13 GB) | `fyp_runs/*/best` (Table VI checkpoints) | complete_pipeline |

Tags not yet copied to your Hub repo are taken from `SRC_FALLBACK` but stored under your repo id, which is the one the notebooks ask for.

In [ ]:
import os
BUNDLE = "base"            # "base" | "regressors" | "flux" | "fyp_runs"  -- one bundle per committed run
DST_REPO = os.environ["HF_REPO_ID"]               # repo id the notebooks request
SRC_FALLBACK = os.environ["HF_FALLBACK_REPO"]     # used for tags missing from DST_REPO
SD = "stable-diffusion-v1-5/stable-diffusion-v1-5"
SD_FILES = ["model_index.json", "vae/config.json", "vae/diffusion_pytorch_model.safetensors",
            "unet/config.json", "unet/diffusion_pytorch_model.safetensors",
            "text_encoder/config.json", "text_encoder/model.safetensors", "tokenizer/*", "scheduler/*"]
BUNDLES = {
    "base":       {SD: SD_FILES,
                   DST_REPO: ["adapted16_final/*", "bridge_final/*", "regressor_v3_final/*", "data_meta/*"]},
    "regressors": {DST_REPO: ["regressor_final/*", "regressor_v2_final/*", "afcrps_final/*"]},
    "flux":       {DST_REPO: ["regressor_modernvae_final/*"],
                   "black-forest-labs/FLUX.1-schnell": ["vae/config.json", "vae/diffusion_pytorch_model.safetensors"]},
    "fyp_runs":   {DST_REPO: ["fyp_runs/*/best/*"]},
}
OUT = "/kaggle/working/sar-assets"

In [ ]:
import os, sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lpips", "torch-fidelity", "huggingface_hub"], check=True)
TOK = None
try:                                     # optional: needed for private repos and the gated FLUX repo
    from kaggle_secrets import UserSecretsClient
    TOK = UserSecretsClient().get_secret("HF_TOKEN") or None
except Exception as e:
    print("no HF_TOKEN secret (fine for public repos):", e)

In [ ]:
import fnmatch, shutil
from huggingface_hub import HfApi, snapshot_download
api, _listing = HfApi(token=TOK), {}

def files_of(repo):
    if repo not in _listing:
        _listing[repo] = api.list_repo_files(repo)
    return _listing[repo]

for repo, pats in BUNDLES[BUNDLE].items():
    dst = f"{OUT}/repos/{repo.replace('/', '__')}"
    for p in pats:
        src = repo if any(fnmatch.fnmatch(f, p) for f in files_of(repo)) else None
        if src is None and repo == DST_REPO and any(fnmatch.fnmatch(f, p) for f in files_of(SRC_FALLBACK)):
            src = SRC_FALLBACK
        if src is None:
            print(f"!! nothing matches {p!r} in {repo}"); continue
        snapshot_download(src, allow_patterns=[p], local_dir=dst, token=TOK)
        print(f"{p:45s} <- {src}")
    shutil.rmtree(f"{dst}/.cache", ignore_errors=True)       # hub bookkeeping, not needed

In [ ]:
if BUNDLE == "base":
    # pip wheels the Kaggle image may lack (installed with --no-deps by the setup cell, only if missing)
    subprocess.run([sys.executable, "-m", "pip", "download", "-q", "--no-deps", "-d", f"{OUT}/wheels",
                    "lpips", "torch-fidelity", "diffusers", "transformers", "accelerate", "safetensors",
                    "torchmetrics", "lightning-utilities", "huggingface_hub", "peft", "kagglehub"], check=True)
    # torch-hub weights fetched on first use: AlexNet/VGG (LPIPS) and Inception (FID)
    os.environ["TORCH_HOME"] = f"{OUT}/torch"
    import lpips
    lpips.LPIPS(net="alex"); lpips.LPIPS(net="vgg")
    from torchmetrics.image.fid import FrechetInceptionDistance
    FrechetInceptionDistance(feature=2048)

In [ ]:
def du(p):
    return sum(os.path.getsize(os.path.join(d, f)) for d, _, fs in os.walk(p) for f in fs) / 1e9
for sub in sorted(os.listdir(OUT)):
    print(f"{sub:10s} {du(os.path.join(OUT, sub)):6.2f} GB")
total = du(OUT)
print(f"TOTAL      {total:6.2f} GB  (bundle '{BUNDLE}')")
assert total < 19.5, "over the ~20 GB output limit: split this bundle"
print(f"Now: Save Version -> Save & Run All, then Output -> New Dataset named 'sar-assets-{BUNDLE}'.")